# Kicker Strategy - Results

In [2]:
import pandas as pd
import numpy as np
import json
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.proportion import proportions_ztest
import scipy.stats as stats

### Data Import

In [3]:
results = glob.glob("results/*.json")
file_list = []

for files in results:
    # Open files
    with open(files, "r") as file:
        data = json.load(file)

    events_list = []
    # Assign filename to "game" column
    game = os.path.basename(files).replace(".json", "")

    # Loop through the runs
    for run in data.get("runs", []):
        run_id = run["run_id"]
        player_1 = run["player_1"]
        player_2 = run["player_2"]
        # Loop through events
        for event in run.get("events", []):
            events_list.append({
                "game": game,
                "run_id": run_id,
                "player_1": player_1,
                "player_2": player_2,
                "timestamp": event["timestamp"],
                "player": event["player"],
                "event_type": event["type"],
                "bar": event.get("bar", pd.NA),
                "side": event.get("side", pd.NA),
                "is_goal": event.get("successful", False) #True if successful, False if not successful
            })

    # Convert to dataframe
    df_game = pd.DataFrame(events_list)
    file_list.append(df_game)

timestamped_data = pd.concat(file_list, ignore_index=True)
timestamped_data = timestamped_data.sort_values(by=["game", "run_id", "timestamp"]).reset_index(drop=True)
timestamped_data["is_shot"] = timestamped_data["event_type"] == "shot"

timestamped_data['round'] = timestamped_data.groupby(['game', 'run_id'])['is_goal'].shift(fill_value=False).cumsum() + 1 #counts up whenever the previous row was a goal (shift function moves it down 1 row)
timestamped_data['is_contact_p1'] = (timestamped_data['player'] == timestamped_data['player_1']) & (timestamped_data['event_type'] == 'contact') #boolean column whether it was a player 1 contact
timestamped_data['is_contact_p2'] = (timestamped_data['player'] == timestamped_data['player_2']) & (timestamped_data['event_type'] == 'contact') #boolean column whether it was a player 2 contact
timestamped_data['is_shot_p1'] = (timestamped_data['player'] == timestamped_data['player_1']) & (timestamped_data['event_type'] == 'shot') #boolean column whether it was a player 1 shot
timestamped_data['is_shot_p2'] = (timestamped_data['player'] == timestamped_data['player_2']) & (timestamped_data['event_type'] == 'shot') #boolean column whether it was a player 2 shot

round_data = timestamped_data.groupby(['game', 'run_id', 'round']).agg( #aggregating the data for each round
    start=('timestamp', 'min'), #minimum timestamp in all of the rows for a round -> minimum -> first row
    end=('timestamp', 'max'), #maximum timestamp in all of the rows for a round -> maximum -> last row
    player_1=('player_1', 'first'), #just takes the value of the first row of a round -> doesn't matter because it's the same in all of the rows
    player_2=('player_2', 'first'), 
    contacts_p1=('is_contact_p1', 'sum'), #sum of all the contacts for player 1 (count of True in the boolean column see above)
    contacts_p2=('is_contact_p2', 'sum'), 
    shots_p1=('is_shot_p1', 'sum'),
    shots_p2=('is_shot_p2', 'sum')
).reset_index()

round_data.columns = ['game', 'run_id', 'round', 'start', 'end', 'player_1', 'player_2', 'contacts_p1', 'contacts_p2', 'shots_p1', 'shots_p2']

winners = timestamped_data[timestamped_data['is_goal']].set_index(['game', 'run_id', 'round'])['player'] #filters the goal rows and then extracts the player who shot the goal (=winner) for each round
round_data = round_data.join(winners, on=['game', 'run_id', 'round']).rename(columns={'player': 'round_winner'}) #joins the winner on game, run_id and round

round_data['duration'] = round_data['end'] - round_data['start'] #calculating duration
round_data['p1_won_round'] = (round_data['round_winner'] == round_data['player_1']) #boolean column whether player 1 won the round
round_data['p2_won_round'] = (round_data['round_winner'] == round_data['player_2']) #boolean columns whether player 2 won the round

round_data['points_p1'] = round_data.groupby(['game', 'run_id'])['p1_won_round'].transform('cumsum') #counts up if player 1 won
round_data['points_p2'] = round_data.groupby(['game', 'run_id'])['p2_won_round'].transform('cumsum') #counts up if player 2 won

round_data.drop(columns=['game','p1_won_round', 'p2_won_round'], inplace=True) #dropping unneccessary columns

### Data Overview

In [4]:
timestamped_data.head()

,game,run_id,player_1,player_2,timestamp,player,event_type,bar,side,is_goal,is_shot,round,is_contact_p1,is_contact_p2,is_shot_p1,is_shot_p2
0,Diana-Hans,0,Diana,Hans,2.985530,Diana,contact,Middle1,Middle,False,False,1,True,False,False,False
1,Diana-Hans,0,Diana,Hans,3.749828,Diana,contact,Attack1,Right,False,False,1,True,False,False,False
2,Diana-Hans,0,Diana,Hans,6.102422,Hans,contact,Attack2,Middle,False,False,1,False,True,False,False
3,Diana-Hans,0,Diana,Hans,10.453081,Hans,shot,NaN,NaN,True,True,1,False,False,False,True
4,Diana-Hans,0,Diana,Hans,20.291921,Diana,contact,Middle1,Middle,False,False,2,True,False,False,False


In [5]:
round_data.head()

,run_id,round,start,end,player_1,player_2,contacts_p1,contacts_p2,shots_p1,shots_p2,round_winner,duration,points_p1,points_p2
0,0,1,2.985530,10.453081,Diana,Hans,2,1,0,1,Hans,7.467551,0,1
1,0,2,20.291921,51.365341,Diana,Hans,5,5,3,2,Diana,31.073420,1,1
2,0,3,56.560030,150.898025,Diana,Hans,15,17,5,7,Diana,94.337994,2,1
3,0,4,164.168779,178.463446,Diana,Hans,3,2,3,1,Diana,14.294667,3,1
4,0,5,191.487753,208.643816,Diana,Hans,4,5,3,2,Hans,17.156064,3,2


## Results

### Hypothesis 1: Olga has a higher shot coversion rate than the rest of the players


H0: There is no statistically significant difference between Olga's shot conversion rate and the conversion rate of the other players.

H1: Olga has a statistically significant higher shot conversion rate compared to the other players.

#### Descriptive results:

In [6]:
player_conversion_rate = timestamped_data.groupby('player').agg(shots=('is_shot','sum'), goals=('is_goal', 'sum')).reset_index()
player_conversion_rate['conversion_rate'] = (player_conversion_rate['goals'] / player_conversion_rate['shots'])
player_conversion_rate.sort_values(by='conversion_rate', ascending=False)

,player,shots,goals,conversion_rate
3,Olga,664,223,0.335843
0,Diana,944,240,0.254237
2,Magnus,1002,186,0.185629
5,Tanja,1141,189,0.165644
4,Simon,1368,219,0.160088
1,Hans,1247,162,0.129912


Olga has a conversion rate of 33.5% -> a third of her shots were successful.

In [7]:
scaler = StandardScaler()
scaler.fit(player_conversion_rate["conversion_rate"].values.reshape(-1, 1))
z_scores = scaler.transform(player_conversion_rate["conversion_rate"].values.reshape(-1, 1))
player_conversion_rate["z_score"] = z_scores
player_conversion_rate.sort_values(by="z_score", ascending=False)

,player,shots,goals,conversion_rate,z_score
3,Olga,664,223,0.335843,1.874807
0,Diana,944,240,0.254237,0.703484
2,Magnus,1002,186,0.185629,-0.281280
5,Tanja,1141,189,0.165644,-0.568126
4,Simon,1368,219,0.160088,-0.647880
1,Hans,1247,162,0.129912,-1.081006


Olga's conversion rate is almost 1.9 standard deviations above the group mean.

#### Statistical results:

We're using a right-tailed Z-test for properties.

We define $p_{\text{Olga}}$ as the shot conversion rate of Olga and $p_{\text{Rest}}$ as the aggregated shot conversion rate of the other players.

$$H_0: p_{\text{Olga}} \le p_{\text{Others}}$$
$$H_1: p_{\text{Olga}} > p_{\text{Others}}$$

In [8]:
olga_goals = timestamped_data.loc[timestamped_data['player'] == 'Olga', 'is_goal'].sum()
olga_shots = timestamped_data.loc[timestamped_data['player'] == 'Olga', 'is_shot'].sum()

others_goals = timestamped_data.loc[timestamped_data['player'] != 'Olga', 'is_goal'].sum()
others_shots = timestamped_data.loc[timestamped_data['player'] != 'Olga', 'is_shot'].sum()

goals = [olga_goals, others_goals]
observations = [olga_shots, others_shots]

z_stat, p_value = proportions_ztest(goals, observations, alternative='larger')

print(f"p value: {p_value:.4e}")

p value: 8.4981e-24


The p-value is below 0.001. There is strong evidence for rejecting the null hypothesis.

### Hypothesis 2: Players who have a attack bias scores a higher number of goals compared to the other players.

**H0**: There is no statistically significant correlation between attack bias and goals scored by players.

**H1**: Players have a statistically significant positive correlation between attack bias and goal scored.

In [9]:
# Extracting attack specific data 
group_cols = ["player", "game"] 
contacts = timestamped_data[timestamped_data["event_type"] == "contact"]
total_contacts = contacts.groupby(group_cols).size()
attack_contact = contacts[contacts["bar"].str.contains("Attack", na=False)].groupby(group_cols).size()  # strips Attack1 to Attack
# We need 2 main  features 'Attack Bias' & 'Total Goals'
attack_bias = (attack_contact / total_contacts).rename("attack_bias").fillna(0)
goals = timestamped_data[(timestamped_data["event_type"] == "shot") & (timestamped_data["is_goal"] == True)]
total_goals = goals.groupby(group_cols).size().rename("total_goals")
player_h2 = pd.concat([attack_bias,total_goals], axis=1).fillna(0)
# Testing the co-relation using stats.pearson

correlation, p_value = stats.pearsonr(player_h2['attack_bias'], player_h2['total_goals'])

print(f"p_value = {p_value:.4f}")
print(correlation)


p_value = 0.5357
0.11768859471281078


In [10]:
player_h2

attack_bias  total_goals
player game                                  
Diana  Diana-Hans       0.225806           50
       Diana-Magnus     0.233937           45
       Diana-Olga       0.257658           50
       Diana-Simon      0.257603           45
       Diana-Tanja      0.227941           50
Hans   Diana-Hans       0.154008           24
       Hans-Magnus      0.221311           35
       Hans-Olga        0.219959           30
       Hans-Simon       0.231237           27
       Hans-Tanja       0.220109           46
Magnus Diana-Magnus     0.172596           33
       Hans-Magnus      0.197115           50
       Magnus-Olga      0.227053           33
       Magnus-Simon     0.207273           35
       Magnus-Tanja     0.204268           35
Olga   Diana-Olga       0.188488           35
       Hans-Olga        0.186457           49
       Magnus-Olga      0.203640           44
       Olga-Simon       0.217687           45
       Olga-Tanja       0.182416           50
Simon  Diana-Simon      0.139183           36
       Hans-Simon       0.153153           50
       Magnus-Simon     0.184874           47
       Olga-Simon       0.171390           42
       Simon-Tanja      0.155828           44
Tanja  Diana-Tanja      0.187402           30
       Hans-Tanja       0.173505           47
       Magnus-Tanja     0.185065           38
       Olga-Tanja       0.215262           35
       Simon-Tanja      0.209268           39

Statistical Results:
_we always look for a p-value < 0.05 to disprove the null hypothesis or to prove or H1. But since here we got a p-value of 0.5357 > 0.05, we reject the Hypothesis H1 and say there is no statistical proof for having a higher attack bias leads to more goals._

### Hypothesis 3: Diana shoots more shots from the left side than the other players

### Hypothesis 4: Hans holds the ball longer before taking a shot compared to the other players


### Hypothesis 5: Tanja has a shorter hold time when she is losing compared to when she is winning

### Hypothesis 6: Shots taken from defense bar or goal bar have a lower goal conversion rate compared to shots from middle/attack bar

### Hypothesis 7: Simon and Magnus have stronger defensive recovery behavior than the other players

### Hypothesis 8: Olga passes the ball more than the other players

### Hypothesis 9: Players with higher defensive recovery rates win more games

### Hypothesis 10: Diana holds the ball shorter before taking a shot compared to the other players

### Hypothesis 11: Magnus wins less against Hans than against anyone else

### Hypothesis 12: The goal success is higher after a pass on the same bar compared to other passes or direct shots

### Hypothesis 13: The goal success is higher after back passes (e.g. Attack -> Middle) than after forward passes or direct shots

### Hypothesis 14: Diana has a higher goal success after a Middle -> Middle pass compared to other passes or direct shots

### Hypothesis 15: Passes have a higher goal success rate than direct shots

### ...